# Module: Confidence Intervals & Bootstrapping

## 1. The Core Intuition: The "Guess My Weight" Dilemma

Imagine a friend asks you, *"How much does that heavy-looking suitcase weigh?"* You have two ways to answer:

1. **The Point Estimate:** *"It weighs exactly 42.3 pounds."* (Highly precise, but almost certainly wrong by at least a few ounces).
2. **The Interval Estimate:** *"I’m pretty confident it’s between 38 and 45 pounds."* (Much more likely to be correct, and honestly acknowledges your uncertainty).

In Data Science, we rarely know the absolute truth about a whole population (e.g., the exact average salary of all data scientists in the world). We only have a **sample** (e.g., a survey of 500 data scientists).

A **Confidence Interval (CI)** is simply the statistical version of option 2. Instead of providing a single, fragile "best guess" (a point estimate like the sample mean), we provide a *range* of plausible values that quantifies our uncertainty.

---

## 2. Demystifying the "95% Confidence" Misconception

When we say, *"We are 95% confident that the true population mean is between X and Y,"* what does that actually mean?

* **The Common Mistake:** "There is a 95% probability that the true population mean lies inside this specific interval." **(False!)**
* **The Ground Truth:** The true population mean is a fixed, unchanging number. It is either inside your interval or it isn't (a probability of 1 or 0).

**The Analogy:** Imagine playing a game of horseshoe pitching. The target stake (the true population parameter) is fixed in the ground. Each sample you take and each confidence interval you calculate is like throwing a horseshoe.

* If you throw 100 horseshoes (calculate 100 different intervals from 100 different samples), **95 of them will successfully ring the stake.** 5 will miss entirely.

Therefore, "95% confidence" describes the **reliability of the process**, not the specific numbers in your final bracket.

---

## 3. Bootstrapping: The Ultimate Statistical "Cheat Code"

Traditionally, finding a confidence interval required rigid mathematical formulas and strict assumptions (e.g., "the data must follow a perfect bell curve"). But what if your data is messy, heavily skewed, or you are calculating something complex like the confidence interval of a *median* or a *ratio*?

Enter **Bootstrapping**. The name comes from the old phrase *"to pull oneself up by one's bootstraps"*—achieving something impossible using only what you already have.

### How it works (The Recipe)

Instead of gathering new data from the real world (which is expensive and time-consuming), we treat our original sample as a mini-universe and simulate new samples from it.

1. **Resample with Replacement:** If your sample has 1000 data points, randomly pick 100 data points from it. Because you sample *with replacement*, some rows will be picked twice, and some won't be picked at all.
2. **Calculate the Statistic:** Compute the metric you care about (e.g., the mean or median) for this new "bootstrap sample".
3. **Repeat:** Do this 1,000 or 10,000 times. You will end up with thousands of different bootstrap means.
4. **Find the Cutoffs:** Sort those thousands of bootstrap means from lowest to highest. For a 95% confidence interval, chop off the bottom 2.5% and the top 2.5%. The values left in the middle form your Confidence Interval!

---

## 4. Let's Code It: Python Implementation

Here is how you implement a bootstrap confidence interval cleanly in Python without relying on heavy statistical libraries.

In [ ]:
import numpy as np

# 1. Simulate some realistic, non-normal data (e.g., website stay times in minutes)
np.random.seed(42)
sample_data = np.random.exponential(scale=5, size=100)
print(f"Sample Mean: {sample_data.mean():.2f} minutes\n")

def bootstrap_ci(data, periods=5000, confidence_level=0.95):
    bootstrap_means = []
    n = len(data)

    # 2. Generate bootstrap replicates
    for _ in range(periods):
        bootstrap_sample = np.random.choice(data, size=n, replace=True) # picking up a sample randomly
        bootstrap_means.append(bootstrap_sample.mean())

    # 3. Calculate percentiles for the interval
    lower_bound = np.percentile(bootstrap_means, (100 - confidence_level * 100) / 2) # (100 - 95) / 2 => 5/2 => 2.5%ile
    upper_bound = np.percentile(bootstrap_means, 100 - (100 - confidence_level * 100) / 2) # (100-2.5) => 97.5%ile

    return lower_bound, upper_bound

lower, upper = bootstrap_ci(sample_data)
print(f"95% Bootstrap Confidence Interval for the Mean:")
print(f"[{lower:.2f} minutes, {upper:.2f} minutes]")

Sample Mean: 4.57 minutes

95% Bootstrap Confidence Interval for the Mean:
[3.68 minutes, 5.51 minutes]


In [ ]:
# print(sample_data)

sample_mean = np.mean(sample_data)

# SE = std / root(n)
se = np.std(sample_data) / np.sqrt(len(sample_data))

print(sample_mean + 1.96 * se)
print(sample_mean - 1.96 * se)

5.470611810992492
3.676868579396223


## 5. Standard Error: Quantifying Sample Variability

The Standard Error (SE) is a measure of the statistical accuracy of an estimate, usually the mean. It indicates how much the sample mean is likely to differ from the true population mean. Unlike standard deviation, which measures the variability within a sample, the standard error measures the variability of *sample means* around the population mean if you were to take many samples.

### How to Calculate Standard Error

The standard error of the mean is typically calculated as:

$SE = \frac{\sigma}{\sqrt{n}}$

Where:
- $\sigma$ is the population standard deviation (often estimated by the sample standard deviation, $s$)
- $n$ is the sample size

In the context of bootstrapping, we can also estimate the standard error as the standard deviation of the bootstrap distribution of the statistic (e.g., the standard deviation of all the `bootstrap_means` we generated).

In [ ]:
import numpy as np

# Re-create sample_data for independent execution of this cell
np.random.seed(42)
sample_data = np.random.exponential(scale=5, size=100)

def calculate_standard_error(data):
    # Using the sample standard deviation as an estimate for population standard deviation
    return np.std(data, ddof=1) / np.sqrt(len(data))

# Calculate and print the standard error of the sample mean
standard_error_mean = calculate_standard_error(sample_data)
print(f"Standard Error of the Sample Mean: {standard_error_mean:.2f} minutes")

Standard Error of the Sample Mean: 0.46 minutes


Practice Notebook using Walmart Data: https://colab.research.google.com/drive/1Eq2cmrOsza2-s0gUmFNevzUpe4G_5X-S?usp=sharing